# Real-time Training Telemetry Demo

This notebook demonstrates the real-time extraction and visualization of mechanistic metrics during a model's training loop using TransformerLens.

By leveraging dynamic dictionary logging alongside the `ActivationCache`, we can isolate the training window where localized phase transitions—such as the formation of induction heads—begin to emerge.

**A note on scaling**: While this 2-layer toy model allows for high-granularity tracking with minimal computational overhead, achieving similar resolution in larger architectures is non-trivial. It requires highly targeted caching and direct manipulation of the telemetry bridge to surface this level of detail without memory exhaustion.

**Compute requirements**: A standard CPU is entirely sufficient for this demonstration. The 500-step training loop will execute rapidly in standard local or cloud-based notebook environments without the need for hardware acceleration.

Initializes the workspace, configures Plotly renderers, and defines the toy model architecture.

**Visualization Context** `Plotly Renderers`:

`Plotly` generates interactive, JavaScript-based visualizations. Google Colab handles these DOM interactions differently than local Jupyter or VS Code environments. We detect the active environment to set the appropriate `plotly.io` renderer (`"colab"` vs `"notebook_connected"`), ensuring the dynamic telemetry plots render correctly without blank output blocks.

**Architectural Rationale:**


* **2 Layers (`n_layers=2`):** The theoretical minimum depth required for induction circuits. Layer 0 creates "previous token" representations, and Layer 1 queries these to predict the next token based on earlier context.

* **2 Heads (`n_heads=2`):** Provides just enough capacity for heads to specialize (e.g., dedicating one head to induction) without creating excessive noise in the telemetry visualizations.

* **GELU Activation (`act_fn="gelu"`):** Selected over ReLU to mirror the smooth non-linearities of modern production LLMs, ensuring the activation dynamics remain representative of real-world architectures.

* **Miniaturized Dimensions:** `d_model=64`, `d_vocab=64`, and `n_ctx=32` are intentionally bottlenecked to force rapid convergence, reliably inducing the phase transition within a brief 500-step training window.

In [1]:
import torch
import numpy as np

# Detect execution environment
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment: Local / Standard Jupyter")

# Environment-specific dependency management
if IN_COLAB:
    %pip install -q transformer_lens
    %pip install -q circuitsvis

import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configure Plotly renderer for correct JavaScript execution
if IN_COLAB:
    pio.renderers.default = "colab"
else:
    pio.renderers.default = "notebook_connected"
print(f"Plotly Renderer: {pio.renderers.default}")

# Must be imported after Colab pip install
from transformer_lens import TransformerBridgeConfig  # noqa: E402
from transformer_lens.model_bridge import TransformerBridge  # noqa: E402

# Configuration
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

# TransformerBridge.boot_native builds a small TL-native model (no HuggingFace
# Hub call, no `transformers` import) and wraps it in a bridge. The bridge's
# `forward`, `run_with_cache`, and `parameters()` surfaces are the same ones
# you'd use on any other bridge-backed model.
cfg = TransformerBridgeConfig(
    n_layers=2,
    d_model=64,
    d_head=32,
    n_heads=2,
    d_mlp=256,
    d_vocab=64,
    n_ctx=32,
    act_fn="gelu",
    normalization_type="LN",
    seed=42,
)
model = TransformerBridge.boot_native(cfg, device=device)


Environment: Local / Standard Jupyter
Plotly Renderer: notebook_connected
Running on cpu


# Attention Telemetry Extraction

This bridge extracts mechanistic metrics directly from the `ActivationCache` during the forward pass.

* **Head Coherence:** Measures attention focus sharpness via normalized entropy. A score of $1.0$ indicates perfect focus on a single token, while $0.0$ indicates uniformly distributed attention.

* **Head Agreement:** Evaluates intra-layer behavioral similarity among attention heads.
* **Variance Normalization:** The agreement metric normalizes against a `0.005` variance constant. This serves as an expected baseline for inter-head variance at the point of specialization in 2-layer models.

**Note:** This constant is a localized architectural assumption. Recalibrating this threshold will likely be necessary when porting the telemetry bridge to larger, higher-dimensional models.

In [2]:
class AttentionTelemetry:
    """Lightweight bridge extracting mechanistic metrics from ActivationCache."""

    @staticmethod
    def compute_metrics(cache, layer_idx):
        """Computes attention coherence and agreement for a given layer.

        Args:
            cache (ActivationCache): The cached activations from the forward pass.
            layer_idx (int): The index of the layer to analyze.

        Returns:
            dict: A dictionary containing layer_idx, head_coherence, and head_agreement.

        Notes on v_max (0.005):
            The agreement normalization constant is derived from the expected inter-head
            attention variance at convergence in 2-layer induction head toy models.
            At convergence, heads specialize (low variance); pre-convergence variance
            peaks near 0.005. This value is task- and architecture-specific; adjust
            if adapting to larger models or different tasks.
        """

        pattern_name = f"blocks.{layer_idx}.attn.hook_pattern"

        # Shape: [batch, heads, seq, seq]
        attn_probs = cache[pattern_name]

        # 1. Head Coherence: 1.0 - (Entropy / Max_Entropy)
        probs = attn_probs + 1e-9
        entropy = -torch.sum(probs * torch.log(probs), dim=-1)  # [batch, heads, seq]
        head_coherence = 1.0 - (entropy.mean(dim=[0, 2]) / np.log(attn_probs.shape[-1]))

        # 2. Head Agreement: 1.0 - clip(Variance / v_max)
        mean_var = torch.var(attn_probs, dim=1).mean()  # Variance across heads
        head_agreement = 1.0 - torch.clamp(mean_var / 0.005, 0.0, 1.0)

        return {
            "layer_idx": layer_idx,
            "head_coherence": head_coherence.mean().item(),
            "head_agreement": head_agreement.item()
        }

# Synthetic Data Generation & Training Loop

This cell generates a highly constrained dataset to force circuit formation and executes the training loop, capturing telemetry at specified intervals.

**A Note on Synthetic Data:**

The repeated sequence generator (`[A, B, C, ..., A, B, C]`) used below is strictly illustrative. It is engineered specifically as a shortcut to force the rapid emergence of in-context look-back circuits (induction heads). It is not meant to serve as an educational standard for model training.

**Transitioning to Real Data:**

Applying this telemetry extraction to real-world datasets requires rigorous attention to detail regarding:

* **Data Quality:** Unstructured noise in the input distribution will severely obscure the mechanistic signals (like coherence and agreement) you are attempting to isolate.

* **Data Type & Tokenization:** Real-world text requires careful handling of padding, EOS/BOS tokens, and sequence packing, all of which dynamically alter attention patterns and can skew your baseline metrics.

* **Ingestion Methodology:** Managing data loaders, batching, and ensuring that telemetry logging steps align with representative samples is critical to preventing metric distortion.

**Performance Optimization:** To prevent the telemetry capture from bottlenecking the training process, `model.run_with_cache` is exclusively executed at logging intervals. Standard forward passes bypass the cache entirely.

In [3]:
def generate_induction_data(batch_size, seq_len, vocab_size, device="cpu"):
    half_len = seq_len // 2
    first_half = torch.randint(0, vocab_size, (batch_size, half_len))
    data = torch.cat([first_half, first_half], dim=1)
    return data.to(device)

# The Impact of Tokenization on Telemetry

When moving from synthetic generators to real-world text, understanding the mechanistic role of special tokens is vital to preventing metric distortion.

**BOS (Begin of Sequence) as an Attention Sink:**

In many autoregressive models, attention heads route their focus to the first token (BOS) when they do not have a strong contextual match elsewhere. This is known as an "attention sink." If unaccounted for, your telemetry will show artificially high **Head Coherence** (because the head is sharply focused on a single token). However, this represents a resting state rather than active circuit engagement.

**EOS (End of Sequence) & Padding:**

Real data requires batching sequences of variable lengths, necessitating padding tokens and EOS markers to denote where a document ends.

* **Masking Failures:** If attention masks are not perfectly aligned with your telemetry extraction, heads might attend to padding tokens, introducing garbage data that drastically skews your **Agreement** metrics.

* **Context Resets:** The transition between unrelated documents (separated by an EOS token) disrupts the contiguous context window. This resets the look-back mechanisms that induction circuits rely on, causing momentary drops in otherwise stable telemetry

# The Training & Real-Time Telemetry Loop

Executing the training sequence while dynamically tracking phase transitions.

**Compute Efficiency (Selective Caching):**

To prevent the extraction process from suffocating the CPU/GPU memory bandwidth, we employ selective caching. Standard forward passes operate normally; `model.run_with_cache` is exclusively invoked at defined logging intervals to extract the telemetry state without severely bottlenecking the training step.

**Rendering Optimization (The Real-Time UI):**

To achieve real-time visualization without crashing the browser's DOM or throttling the PyTorch loop:

1. **Memory Pre-allocation:** We pre-allocate `NaN` arrays for the telemetry traces, completely bypassing costly array reallocation during the loop.

2. **In-Place Mutation:** Instead of generating hundreds of static Plotly objects, we mutate the figure's trace data directly and use `IPython.display.clear_output` to cleanly redraw the frame in the exact same output block.

3. **Static Fallback:** *If you wish to bypass real-time rendering to maximize training speed, simply comment out the `clear_output(wait=True)` and `fig.show()` lines inside the loop, and call `fig.show()` once at the very end of the cell.

**Mechanistic Observation (The Phase Transition):**

Watch the dual-plot for the localized phase transition: a distinct window where the model suddenly "discovers" the induction algorithm. This is marked by a violent crash in the loss curve and a simultaneous, sharp spike in the last layer's Attention Coherence.

In [4]:
from IPython.display import clear_output
import numpy as np  # noqa: F811
import torch

# --- 1. Self-Contained Synthetic Data Generator ---
def get_batch(batch_size=16, seq_len=model.cfg.n_ctx, vocab_size=model.cfg.d_vocab):
    """Generates repeated sequences [A, B, C, A, B, C] to force induction circuitry."""
    half_len = seq_len // 2
    random_tokens = torch.randint(0, vocab_size, (batch_size, half_len), device=device)
    return torch.cat([random_tokens, random_tokens], dim=1)

# --- 2. Pre-allocate Memory for Real-Time Plotting ---
TOTAL_STEPS = 500
LOG_INTERVAL = 10
num_logging_steps = TOTAL_STEPS // LOG_INTERVAL

logged_steps = np.arange(0, TOTAL_STEPS, LOG_INTERVAL)

loss_data = np.full(num_logging_steps, np.nan)
coherence_data = np.full(num_logging_steps, np.nan)
heatmap_data = np.full((model.cfg.n_layers, num_logging_steps), np.nan)

# --- 3. Initialize the Plotly Figure ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Phase Transition: Loss Crash vs. Circuit Formation",
        "Attention Coherence Heatmap by Layer Depth"
    ),
    specs=[[{"secondary_y": True}], [{"type": "heatmap"}]]
)

# Trace 0: Loss Curve
fig.add_trace(
    go.Scatter(x=logged_steps, y=loss_data, name="Loss (CE)", line=dict(color='gray', dash='dash')),
    row=1, col=1, secondary_y=False
)

# Trace 1: Last Layer Coherence Curve
last_layer_idx = model.cfg.n_layers - 1
fig.add_trace(
    go.Scatter(x=logged_steps, y=coherence_data, name=f"Layer {last_layer_idx} Coherence", line=dict(color='#1f77b4', width=2.5)),
    row=1, col=1, secondary_y=True
)

# Trace 2: Layer Heatmap
fig.add_trace(
    go.Heatmap(
        z=heatmap_data, x=logged_steps, y=[f"L{i}" for i in range(model.cfg.n_layers)],
        colorscale='Magma', zmin=0.0, zmax=1.0,
        colorbar=dict(title="Coherence (0-1)", orientation='h', y=-0.25, len=0.5)
    ),
    row=2, col=1
)

fig.update_layout(height=700, template="plotly_white", margin=dict(t=50, b=50))
fig.update_yaxes(title_text="Cross Entropy Loss", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="Coherence", secondary_y=True, range=[0, 1.1], row=1, col=1)
fig.update_xaxes(range=[0, TOTAL_STEPS])
# --- 4. The Training & Telemetry Loop ---
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

log_idx = 0
for step in range(TOTAL_STEPS):
    batch = get_batch()

    # Selective Caching Strategy
    if step % LOG_INTERVAL == 0:
        loss, cache = model.run_with_cache(batch, return_type="loss")

        # Update baseline data arrays
        loss_data[log_idx] = loss.item()

        # Extract mechanistic metrics using the static method from Cell 3
        for layer in range(model.cfg.n_layers):
            layer_metrics = AttentionTelemetry.compute_metrics(cache, layer)
            heatmap_data[layer, log_idx] = layer_metrics['head_coherence']

            # Specifically grab the last layer for the line graph
            if layer == last_layer_idx:
                coherence_data[log_idx] = layer_metrics['head_coherence']

        # Mutate Plotly traces in-place
        fig.data[0].x = logged_steps
        fig.data[0].y = loss_data

        fig.data[1].x = logged_steps
        fig.data[1].y = coherence_data

        fig.data[2].x = logged_steps
        fig.data[2].z = heatmap_data

        # Redraw the UI
        clear_output(wait=True)
        fig.show()

        log_idx += 1
    else:
        # Standard forward pass (bypassing the cache for speed)
        loss = model(batch, return_type="loss")

    # Standard PyTorch Optimization
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()